# Run All Experiments

**Paper**: Engineering Truthiness: A Standard for Pseudo–Ground Truth in Machine Learning Evaluation

This notebook executes all four experiments sequentially and reports results.
Each experiment automatically falls back to synthetic data if full dependencies
(torch, datasets, snorkel) are unavailable.

**Options**: Set `EXPERIMENTS` below to run a subset, e.g. `[1, 3]`.

---

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
# Which experiments to run (1-4, or any subset)
EXPERIMENTS = [1, 2, 3, 4]

# Set to True for a parse-only check (no execution)
DRY_RUN = False


In [ ]:
import json, os, sys, time
from pathlib import Path

os.environ['MPLBACKEND'] = 'Agg'

# Resolve paths
_here = Path('.').resolve()
REPO_ROOT = _here.parent if _here.name == 'notebooks' else _here
NB_DIR = REPO_ROOT / 'notebooks'

NOTEBOOKS = {
    1: '01_synthetic_theory_validation.ipynb',
    2: '02_text_weak_supervision.ipynb',
    3: '03_clinical_jaccard_fairness.ipynb',
    4: '04_cifar_vision_stress.ipynb',
}
DESCRIPTIONS = {
    1: 'Synthetic Double-Noise Validation (Theorem 1)',
    2: 'Weakly Supervised Text Classification (IMDB/Snorkel)',
    3: 'Clinical Trial Matching (Jaccard Silver Labels)',
    4: 'Vision Benchmark Stress Test (CIFAR-10)',
}

print(f'Repo root: {REPO_ROOT}')
print(f'Notebooks: {NB_DIR}')
print(f'Experiments: {EXPERIMENTS}')


In [ ]:
def run_notebook(nb_path, repo_root):
    """Execute all code cells from a notebook."""
    with open(nb_path) as f:
        nb = json.load(f)

    cells = []
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        src = ''.join(cell['source']).strip()
        if not src or src.startswith('!'):
            continue
        if 'IPython.display' in src or 'ipy_display' in src:
            continue
        cells.append(src)

    old_cwd = os.getcwd()
    os.chdir(repo_root / 'notebooks')

    namespace = {'__name__': '__main__'}
    errors = []
    t0 = time.time()

    for i, src in enumerate(cells):
        try:
            exec(src, namespace)
        except Exception as e:
            errors.append(f'Cell {i}: {type(e).__name__}: {e}')
            print(f'  [WARN] Cell {i}: {type(e).__name__}: {e}')

    elapsed = time.time() - t0
    os.chdir(old_cwd)

    return {
        'status': 'PASS' if not errors else 'PARTIAL',
        'cells': len(cells),
        'errors': errors,
        'elapsed': round(elapsed, 1),
    }

print('Runner function defined.')


In [ ]:
# ==========================================
# EXECUTE ALL EXPERIMENTS
# ==========================================
results = {}
total_t0 = time.time()

for exp in sorted(EXPERIMENTS):
    nb_name = NOTEBOOKS[exp]
    nb_path = NB_DIR / nb_name
    desc = DESCRIPTIONS[exp]

    print(f'\n{"="*60}')
    print(f'  [{exp}/4] {desc}')
    print(f'  {nb_name}')
    print(f'{"="*60}')

    if DRY_RUN:
        with open(nb_path) as f:
            n_cells = sum(1 for c in json.load(f)['cells'] if c['cell_type'] == 'code')
        results[exp] = {'status': 'DRY_RUN', 'cells': n_cells}
        print(f'  [DRY RUN] {n_cells} code cells')
        continue

    r = run_notebook(nb_path, REPO_ROOT)
    results[exp] = r
    icon = '\u2713' if r['status'] == 'PASS' else '\u26A0'
    print(f'  {icon} {r["status"]} — {r["cells"]} cells in {r["elapsed"]}s')

total = time.time() - total_t0

print(f'\n{"="*60}')
print(f'  SUMMARY — {total:.1f}s total')
print(f'{"="*60}')
for exp in sorted(results):
    r = results[exp]
    icon = {'PASS': '\u2713', 'PARTIAL': '\u26A0', 'DRY_RUN': '\u25CB'}.get(r['status'], '?')
    print(f'  {icon}  Exp {exp}: {DESCRIPTIONS[exp]} — {r["status"]}')

# Count outputs
out_dir = REPO_ROOT / 'outputs'
if out_dir.exists():
    figs = list(out_dir.rglob('*.png'))
    csvs = list(out_dir.rglob('*.csv'))
    print(f'\n  Outputs: {len(figs)} figures, {len(csvs)} tables')
    print(f'  Location: {out_dir}/')
